<p class="lead">A prediction without an uncertainty is not a result, it is a number. This unit builds the three things a referee of a QSAR paper will ask for and a colleague about to spend money on a compound actually needs: an honest statement of <em>what the model cannot know</em> (aleatoric versus epistemic uncertainty, and the experimental error floor of the data itself), a definition of <em>where the model may be used</em> (the applicability domain, computed four ways and tested by how much accuracy degrades outside each), and an interval or prediction set with a <em>guaranteed</em> error rate (inductive conformal prediction, written here by hand in a dozen lines of R). The central experiment is the one that decides whether such guarantees survive contact with chemistry: the same conformal predictor calibrated and tested under a random split, and then under a scaffold split, where the exchangeability assumption that makes it valid no longer holds.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad trata la cuantificación de la incertidumbre. Se distingue la incertidumbre aleatoria de la epistémica y se mide el piso de error experimental a partir de las mediciones replicadas del propio conjunto de datos. Se definen y calculan cuatro dominios de aplicabilidad (caja delimitadora de descriptores, apalancamiento con el umbral h* y el gráfico de Williams, distancia de Tanimoto a los k vecinos más cercanos y densidad por núcleos escrita a mano) y se muestra cuánto se degrada la exactitud fuera de cada uno. Se construyen intervalos de predicción con la dispersión del ensamble y con bosques de regresión cuantílica, verificando su cobertura empírica, y luego se implementa a mano la predicción conformal inductiva, para regresión (simple y normalizada por dificultad) y para clasificación (conjuntos de predicción y la variante Mondrian condicionada por clase), comprobando la cobertura al 80, 90 y 95 por ciento bajo partición aleatoria, en veinte repeticiones y bajo partición por andamiaje. Se calibran probabilidades con Platt e isotónica y se cierra con los cinco principios de validación de la OCDE y un informe tipo QMRF generado por el propio documento.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Distinguish aleatoric from epistemic uncertainty and explain why a point prediction without uncertainty is incomplete.</li>
<li>Define and compute an applicability domain (descriptor ranges, leverage and the Williams plot, distance to training set, density) and report how accuracy degrades outside it.</li>
<li>Obtain prediction intervals and class-probability sets from ensembles, quantile regression forests and inductive conformal prediction, and verify their empirical coverage.</li>
<li>Calibrate predicted probabilities (Platt, isotonic) and read a reliability diagram and a Brier score.</li>
<li>Comply with the OECD principles for QSAR model validation and fill a QMRF-style report for a model you fitted.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Run every chunk; every number in the prose is printed above it. In a fresh Google Colab R runtime run <code>install.packages(c("data.table","ranger","glmnet","patchwork"))</code> once before the first chunk. The model used throughout is refitted here in about a minute.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(Matrix); library(ranger); library(glmnet)
  library(ggplot2);    library(patchwork)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10), panel.grid.minor = element_blank()))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| ranger", as.character(packageVersion("ranger")),
    "| glmnet", as.character(packageVersion("glmnet")),
    "| ggplot2", as.character(packageVersion("ggplot2")), "\n")


In [ ]:
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
dp <- function(f) data_file(f)
df  <- fread(dp("rock1_chembl_curated.csv")); cmp <- df[censored == FALSE]
DESC <- c("MolWt","MolLogP","TPSA","HBD","HBA","RotB",
          "AromaticRings","RingCount","HeavyAtoms","FractionCSP3","QED")
Xd <- as.matrix(cmp[, ..DESC]); y <- cmp$pActivity; ycls <- as.integer(y >= 7); scaf <- cmp$scaffold_smiles
sp <- fread(dp("rock1_morgan_sparse.csv.gz")); keep <- sp[id %in% cmp$inchikey]
FP <- sparseMatrix(i = match(keep$id, cmp$inchikey), j = keep$bit + 1L, x = 1, dims = c(nrow(cmp), 2048))
Xfp <- as.matrix(FP); colnames(Xfp) <- paste0("b", 0:2047); storage.mode(Xfp) <- "double"
random_split <- function(n, seed = 0, fr = c(0.6, 0.2)) {
  set.seed(seed); p <- sample(n); a <- floor(fr[1]*n); b <- floor(sum(fr)*n)
  list(tr = p[1:a], ca = p[(a + 1):b], te = p[(b + 1):n]) }
scaffold_split <- function(s, fr = c(0.6, 0.2)) {   # largest Murcko families to training
  g <- split(seq_along(s), s); o <- order(-lengths(g), names(g))
  n <- length(s); ntr <- floor(fr[1]*n); nca <- floor(fr[2]*n); tr <- ca <- te <- integer(0)
  for (k in o) { i <- g[[k]]
    if (length(tr) < ntr) tr <- c(tr, i) else if (length(ca) < nca) ca <- c(ca, i) else te <- c(te, i) }
  list(tr = tr, ca = ca, te = te) }
R_ <- random_split(nrow(Xd)); S_ <- scaffold_split(scaf)
TR <- R_$tr; CA <- R_$ca; TE <- R_$te; TRs <- S_$tr; CAs <- S_$ca; TEs <- S_$te
cat(sprintf("compounds %d; potent (pActivity >= 7) %.1f%%\n", length(y), 100*mean(ycls)))
cat(sprintf("random split    : train %d / calibration %d / test %d\n", length(TR), length(CA), length(TE)))
cat(sprintf("scaffold split  : train %d / calibration %d / test %d; scaffolds shared between train and test: %d\n",
            length(TRs), length(CAs), length(TEs), length(intersect(scaf[TRs], scaf[TEs]))))



Three parts, not two: **training** (fit the model), **calibration** (fit the uncertainty) and **test** (measure whether the uncertainty was right). The calibration part is what makes conformal prediction possible and is the piece most pipelines forget.

## 1. Two kinds of uncertainty, and the floor underneath both

A prediction can be uncertain for two different reasons<sup class="c"><a href="#r1">[1]</a></sup>.

**Aleatoric uncertainty** is noise in the world (or in the measurement of it) that no amount of extra data removes. For bioactivity data it is the irreducible scatter between two laboratories measuring the same compound in the same assay type: different cell batches, different ATP concentrations, different curve-fitting software. It is a property of the *target variable*.

**Epistemic uncertainty** is ignorance of the model: the compound is unlike anything in the training set, so the model is extrapolating. It *does* shrink with more (relevant) data, and it is the kind an applicability domain is designed to detect.

The two require different treatments and both set limits on what any model can achieve. The aleatoric part can be measured directly here, because the curated table keeps replicate information.



In [ ]:
rep_ <- cmp[n_measurements > 1]
pooled_sd <- sqrt(sum((rep_$n_measurements - 1)*rep_$pActivity_sd^2, na.rm = TRUE) /
                  sum(rep_$n_measurements - 1, na.rm = TRUE))
cat(sprintf("compounds with replicate measurements: %d of %d (%.0f measurements in total)\n",
            nrow(rep_), nrow(cmp), sum(rep_$n_measurements)))
cat(sprintf("within-compound spread: median SD %.3f, mean SD %.3f, 90th percentile %.3f log units\n",
            median(rep_$pActivity_sd, na.rm = TRUE), mean(rep_$pActivity_sd, na.rm = TRUE),
            quantile(rep_$pActivity_sd, 0.9, na.rm = TRUE)))
cat(sprintf("pooled within-compound standard deviation sigma_exp = %.3f log units\n", pooled_sd))
cat(sprintf("-> a perfect model predicting a single-measurement compound would still have RMSE >= %.3f; comparing two independent measurements, %.3f\n",
            pooled_sd, pooled_sd*sqrt(2)))
cat(sprintf("compounds flagged wide_spread (replicate range > 1 log unit): %d; among the replicated compounds %d of %d (%.1f%%)\n",
            sum(cmp$wide_spread), sum(rep_$pActivity_range > 1, na.rm = TRUE), nrow(rep_),
            100*mean(rep_$pActivity_range > 1, na.rm = TRUE)))


In [ ]:
g1 <- ggplot(rep_[!is.na(pActivity_sd)], aes(pActivity_sd)) +
  geom_histogram(bins = 40, fill = PAL["blue"], colour = NA) +
  geom_vline(xintercept = pooled_sd, colour = PAL["orange"], linewidth = 1) +
  labs(x = "within-compound SD of replicate measurements (log units)", y = "compounds",
       title = sprintf("Replicate spread (%d compounds), pooled SD = %.2f", nrow(rep_), pooled_sd))
g2 <- ggplot(rep_[!is.na(pActivity_range)], aes(pActivity_range)) +
  geom_histogram(bins = 40, fill = PAL["green"], colour = NA) +
  geom_vline(xintercept = 1, colour = PAL["orange"], linetype = 2) +
  labs(x = "range of replicate measurements (log units)", y = "compounds",
       title = "Range between the extreme measurements")
print(g1 | g2)
cat(sprintf("median replicate range %.3f, 90th percentile %.3f log units\n",
            median(rep_$pActivity_range, na.rm = TRUE), quantile(rep_$pActivity_range, .9, na.rm = TRUE)))



<p class="caption"><b>Figure 1.</b> Left: the standard deviation of the replicate measurements of each multiply-measured compound, with the pooled estimate marked. Most compounds agree exactly — the median within-compound SD is 0.000, because most replicated compounds were measured twice with the same reported value — but a tail disagrees by a log unit or more between laboratories, and the pooled value is dominated by that tail. Right: the range between the extreme measurements; the dashed line is the 1-log-unit threshold the curated table uses for its wide_spread flag.</p>

A pooled within-compound standard deviation of about half a log unit is in line with the published analyses of public bioactivity data: Kramer and co-workers estimated about 0.54 log units for heterogeneous Ki values<sup class="c"><a href="#r2">[2]</a></sup> and Kalliokoski and co-workers about 0.68 for mixed IC50 data<sup class="c"><a href="#r3">[3]</a></sup>. **This is the floor.** A model whose test RMSE approaches 0.7 log units on such data is not mediocre; it is close to the measurement noise, and the remaining error cannot be removed by a better algorithm — only by better experiments.

<div class="box bridge"><h4>Domain context — what an IC50 is, and why two laboratories disagree</h4>
<p>An <strong>IC50</strong> is the concentration of compound at which an assay signal falls to half of its uninhibited value; a <strong>Ki</strong> is a thermodynamic binding constant derived from it; both are converted here to <strong>pActivity</strong> = -log10(molar concentration), so 7.0 means 100 nM and one unit is a tenfold change in potency. The value depends on the assay: for an ATP-competitive kinase inhibitor the measured IC50 shifts with the ATP concentration used, and functional (cell-based) assays give systematically different numbers from biochemical ones. ChEMBL records this provenance, and the curated table aggregated repeated measurements per compound while keeping their spread. Censored records ("&gt; 10 µM") were deliberately kept in the curated file and are excluded here, because a censored value is a different kind of uncertainty again — an interval, not a point.</p></div>

## 2. The model, its two splits, and where the error comes from

Everything below uses one model: a random forest<sup class="c"><a href="#r29">[29]</a></sup> on 2,048-bit Morgan fingerprints of the curated ChEMBL table<sup class="c"><a href="#r28">[28]</a></sup>, fitted with `ranger`<sup class="c"><a href="#r31">[31]</a></sup> and refitted inside this document. Two splits are used throughout: a random split, and a deterministic Bemis-Murcko scaffold split<sup class="c"><a href="#r27">[27]</a></sup> in which no scaffold family appears on both sides — the standard stress test for "would this model work on a new chemotype"<sup class="c"><a href="#r26">[26]</a><a href="#r34">[34]</a></sup>.

<div class="box bridge"><h4>An R trap worth knowing: <code>ranger(seed = 0)</code> is not reproducible</h4>
<p><code>ranger</code>'s <code>seed</code> argument documents 0 as "ignore the R seed": passing <code>seed = 0</code> makes every fit different, silently, even inside a <code>set.seed(0)</code> script. Two identical calls with <code>seed = 0</code> on this data gave test predictions differing by up to 0.2 log units. Every forest in this unit therefore uses <code>seed = 1</code>, which is reproducible and, as a bonus, independent of <code>num.threads</code>. The same trap exists in other C++-backed R packages: check whether the package has its own seed argument before trusting <code>set.seed</code> alone.</p></div>



In [ ]:
fit_rf <- function(tr, nt = 300) ranger(x = Xfp[tr, ], y = y[tr], num.trees = nt,
                                        min.node.size = 2, seed = 1, num.threads = 4)
pr <- function(m, idx) predict(m, data = Xfp[idx, ])$predictions
nbits <- rowSums(FP)
knn_dist <- function(q, tr, k = 5) {              # mean Tanimoto distance to the k nearest training rows
  inter <- as.matrix(tcrossprod(FP[q, , drop = FALSE], FP[tr, , drop = FALSE]))
  S <- inter/(outer(nbits[q], nbits[tr], "+") - inter)
  apply(S, 1, function(r) 1 - mean(sort(r, decreasing = TRUE)[1:k])) }
m_rand <- fit_rf(TR); m_scaf <- fit_rf(TRs)
info <- do.call(rbind, list(
  data.frame(split = "random", n_train = length(TR), n_test = length(TE),
             R2 = 1 - sum((y[TE] - pr(m_rand, TE))^2)/sum((y[TE] - mean(y[TE]))^2),
             MAE = mean(abs(y[TE] - pr(m_rand, TE))), RMSE = sqrt(mean((y[TE] - pr(m_rand, TE))^2)),
             median_knn_distance = median(knn_dist(TE, TR))),
  data.frame(split = "scaffold", n_train = length(TRs), n_test = length(TEs),
             R2 = 1 - sum((y[TEs] - pr(m_scaf, TEs))^2)/sum((y[TEs] - mean(y[TEs]))^2),
             MAE = mean(abs(y[TEs] - pr(m_scaf, TEs))), RMSE = sqrt(mean((y[TEs] - pr(m_scaf, TEs))^2)),
             median_knn_distance = median(knn_dist(TEs, TRs)))))
knitr::kable(info, digits = 3, row.names = FALSE, caption = "One model, two splits")
cat(sprintf("experimental floor sigma_exp = %.3f; the random-split RMSE is %.2f times it, the scaffold-split RMSE %.2f times it\n",
            pooled_sd, info$RMSE[1]/pooled_sd, info$RMSE[2]/pooled_sd))



The two splits differ in exactly the way a validation unit predicts, and the distance column explains why: under a scaffold split the median test compound is much further from the training set in fingerprint space, and the error grows accordingly. The random-split RMSE sits at a small multiple of the experimental floor; most of the remaining error is epistemic and should be concentrated in the compounds the model has never seen the like of. Quantifying *that* is the applicability domain — and Section 3 finds that the quantification is harder than the literature suggests.

## 3. The applicability domain, four ways

The OECD guidance makes "a defined domain of applicability" the third of its five validation principles, without prescribing a method; the literature offers a dozen<sup class="c"><a href="#r4">[4]</a></sup>. Four are computed below, in increasing order of usefulness for a fingerprint model.

**(a) Bounding box (descriptor ranges).** A compound is inside if every descriptor lies within the training minimum and maximum. Crude, and generous: a molecule can satisfy all eleven ranges separately while occupying a region none of the training compounds inhabit.

**(b) Leverage and the Williams plot.** For a linear model with design matrix \(\mathbf{X}\), the leverage of a query point is \(h_i = \mathbf{x}_i^{\top}(\mathbf{X}^{\top}\mathbf{X})^{-1}\mathbf{x}_i\), the diagonal of the hat matrix. The conventional warning threshold is \(h^{*} = 3p/n\), and the **Williams plot** of standardised residual against leverage is the classical QSAR diagnostic: the top-right quadrant holds influential outliers<sup class="c"><a href="#r5">[5]</a><a href="#r6">[6]</a></sup>.

**(c) Distance to the training set.** The mean Tanimoto distance to the \(k\) nearest training compounds, in the same fingerprint space the model uses<sup class="c"><a href="#r7">[7]</a></sup>.

**(d) Density.** A kernel density estimate of the training distribution (here in the first five principal components of the standardised descriptors, with a Gaussian kernel written out in two lines); a compound in a low-density region is an extrapolation even if it is inside the bounding box. The training densities are computed **leave-one-out**, without the point's own kernel, so that the threshold is comparable with the test values.

A domain definition is only worth reporting if accuracy actually degrades outside it, so each is scored that way.



In [ ]:
pred_te <- pr(m_rand, TE); err <- abs(y[TE] - pred_te)
mu <- colMeans(Xd[TR, ]); sg <- apply(Xd[TR, ], 2, sd)
Ztr <- scale(Xd[TR, ], mu, sg); Zte <- scale(Xd[TE, ], mu, sg)
lo <- apply(Xd[TR, ], 2, min); hi <- apply(Xd[TR, ], 2, max)
in_box <- apply(Xd[TE, ], 1, function(r) all(r >= lo & r <= hi))
d_knn <- knn_dist(TE, TR); thr_knn <- quantile(knn_dist(CA, TR), 0.90)   # threshold from calibration
pc <- prcomp(Ztr); P5tr <- pc$x[, 1:5]; P5te <- predict(pc, Zte)[, 1:5]
kde <- function(Q, Ref, h = 0.8, loo = FALSE) apply(Q, 1, function(q) {
  k <- exp(-rowSums(sweep(Ref, 2, q)^2)/(2*h^2))
  log((sum(k) - loo*1)/ (nrow(Ref) - loo)) })
dens_te <- kde(P5te, P5tr); thr_dens <- quantile(kde(P5tr, P5tr, loo = TRUE), 0.05)
ad <- do.call(rbind, lapply(list(list("bounding box (descriptors)", in_box),
                                 list(sprintf("kNN(5) Tanimoto distance < %.2f", thr_knn), d_knn < thr_knn),
                                 list("kernel density above the 5th percentile", dens_te >= thr_dens)),
  function(z) { inside <- z[[2]]
    data.frame(AD_definition = z[[1]], pct_inside = 100*mean(inside),
               MAE_inside = mean(err[inside]), MAE_outside = mean(err[!inside]),
               RMSE_inside = sqrt(mean(err[inside]^2)), RMSE_outside = sqrt(mean(err[!inside]^2))) }))
knitr::kable(ad, digits = 3, row.names = FALSE, caption = "Three applicability domains, scored by degradation")
cat(sprintf("Spearman correlation with the absolute error: kNN distance %+.3f, negative log-density %+.3f\n",
            cor(err, d_knn, method = "spearman"), cor(err, -dens_te, method = "spearman")))


In [ ]:
qt <- cut(d_knn, quantile(d_knn, 0:4/4), include.lowest = TRUE,
          labels = c("Q1 nearest", "Q2", "Q3", "Q4 farthest"))
deg <- data.frame(quartile = levels(qt), n = as.integer(table(qt)),
                  mean_distance = tapply(d_knn, qt, mean), MAE = tapply(err, qt, mean),
                  RMSE = sqrt(tapply(err^2, qt, mean)), frac_error_gt_1 = tapply(err > 1, qt, mean))
knitr::kable(deg, digits = 3, row.names = FALSE,
             caption = "Error against distance to the training set, in quartiles")
cat(sprintf("MAE goes from %.3f in the nearest quartile to %.3f in the farthest; the share of errors above 1 log unit from %.1f%% to %.1f%%\n",
            deg$MAE[1], deg$MAE[4], 100*deg$frac_error_gt_1[1], 100*deg$frac_error_gt_1[4]))


In [ ]:
set.seed(0); cvr <- cv.glmnet(Ztr, y[TR], alpha = 0, nfolds = 10)
res <- y[TE] - as.numeric(predict(cvr, newx = Zte, s = "lambda.min")); std_res <- res/sd(res)
Xmat <- cbind(1, Ztr); Hinv <- solve(crossprod(Xmat)); Xq <- cbind(1, Zte)
lev <- rowSums((Xq %*% Hinv) * Xq)
p_param <- ncol(Xmat); h_star <- 3*p_param/nrow(Ztr); out_lev <- lev > h_star
cat(sprintf("ridge model on %d descriptors, n_train = %d  ->  h* = 3p/n = %.4f\n",
            p_param - 1, nrow(Ztr), h_star))
cat(sprintf("test compounds with leverage above h*: %d (%.1f%%)\n", sum(out_lev), 100*mean(out_lev)))
cat(sprintf("MAE inside the leverage domain %.3f, outside %.3f\n",
            mean(abs(res[!out_lev])), mean(abs(res[out_lev]))))
cat(sprintf("standardised residuals beyond +-3: %d compounds (%.1f%%); top-right quadrant (|r| > 3 and h > h*): %d\n",
            sum(abs(std_res) > 3), 100*mean(abs(std_res) > 3), sum(abs(std_res) > 3 & out_lev)))


In [ ]:
g1 <- ggplot(data.frame(lev, std_res), aes(lev, std_res)) + geom_point(size = .7, colour = PAL["blue"], alpha = .7) +
  geom_vline(xintercept = h_star, linetype = 2, colour = PAL["orange"]) +
  geom_hline(yintercept = c(-3, 3), linetype = 3, colour = PAL["orange"]) +
  labs(x = "leverage h", y = "standardised residual",
       title = sprintf("Williams plot (ridge, h* = %.3f)", h_star))
g2 <- ggplot(deg, aes(factor(quartile, levels = quartile), MAE)) +
  geom_col(fill = PAL["blue"], alpha = .85) +
  geom_hline(yintercept = pooled_sd, linetype = 2, colour = PAL["orange"]) +
  annotate("text", x = 1.5, y = pooled_sd*1.04, label = "experimental floor", size = 2.4, colour = PAL["orange"]) +
  labs(x = NULL, y = "MAE (log units)", title = "Error vs distance to the training set") +
  theme(axis.text.x = element_text(size = 7))
bins <- quantile(d_knn, seq(0, 1, length.out = 11))
cent <- (head(bins, -1) + tail(bins, -1))/2
binned <- sapply(1:10, function(i) mean(err[d_knn >= bins[i] & d_knn <= bins[i + 1]]))
g3 <- ggplot(data.frame(d = d_knn, e = err), aes(d, e)) + geom_point(size = .5, colour = "grey65", alpha = .6) +
  geom_line(data = data.frame(d = cent, e = binned), colour = PAL["orange"], linewidth = 1) +
  geom_point(data = data.frame(d = cent, e = binned), colour = PAL["orange"], size = 1.2) +
  labs(x = "mean Tanimoto distance to 5 nearest training compounds", y = "|error| (log units)",
       title = "The same relationship, unbinned")
print(g1 | g2 | g3)



<p class="caption"><b>Figure 2.</b> Left: Williams plot for the descriptor ridge model — leverage against standardised residual, with the h* threshold (dashed) and the ±3 residual lines (dotted). Centre: mean absolute error of the fingerprint model by quartile of distance to the training set, with the experimental error floor marked; the relationship is not monotone. Right: the same data unbinned with a decile-averaged trend; the cloud is wide at every distance, so distance explains very little of the error.</p>

The table and the figure together give an uncomfortable but useful result, and it is more negative than the Python edition's.

- The **descriptor bounding box** declares almost every test compound inside, and the handful outside it are barely predicted worse. As an applicability domain for a fingerprint model it is useless: it is measured in the wrong space.
- **Leverage** behaves as the QSAR literature says — *for the model it is defined for*. For the ridge model, the compounds beyond \(h^{*}\) are predicted worse. It says nothing about the random forest, whose geometry is not that of a linear model in descriptor space.
- **kNN Tanimoto distance** separates the extreme quartiles by a little, but the relationship is **not monotone**: the *nearest* quartile has a mean absolute error as large as the farthest and the largest share of errors above one log unit. The Spearman correlation between distance and absolute error is essentially zero. The explanation is chemical rather than statistical — the nearest quartile is full of close analogues of training compounds, which is exactly where **activity cliffs** live, and a cliff is a large error at zero distance.
- **Kernel density** in descriptor-PCA space adds little once distance is known, because it measures the same thing more crudely.

### 3.1 Retention curves: the operational form of an applicability domain

A project does not ask "is this compound inside the domain" but "if I only trust the most reliable half of my predictions, how good are they?". The answer is a **retention curve**: sort the test set by an uncertainty proxy, keep the most reliable fraction, and plot the error of what is kept.



In [ ]:
m_all <- ranger(x = Xfp[TR, ], y = y[TR], num.trees = 300, min.node.size = 2, seed = 1, num.threads = 4)
sd_trees <- apply(predict(m_all, data = Xfp[TE, ], predict.all = TRUE)$predictions, 1, sd)
set.seed(3)
proxies <- list("kNN(5) Tanimoto distance" = d_knn, "negative kernel density" = -dens_te,
                "ensemble spread (tree SD)" = sd_trees, "random order (control)" = rnorm(length(TE)))
fr <- seq(0.1, 1, length.out = 19)
cur <- do.call(rbind, lapply(names(proxies), function(nm) {
  o <- order(proxies[[nm]])
  data.frame(proxy = nm, frac = 100*fr,
             rmse = sapply(fr, function(f) sqrt(mean(err[o[1:max(round(f*length(TE)), 10)]]^2)))) }))
print(ggplot(cur, aes(frac, rmse, colour = proxy)) + geom_line() + geom_point(size = .8) +
        geom_hline(yintercept = pooled_sd, linetype = 2) +
        scale_colour_manual(values = unname(PAL)) +
        labs(x = "% of test compounds retained (most reliable first)",
             y = "RMSE of the retained set (log units)", title = "Retention curves") +
        theme(legend.position = "bottom", legend.title = element_blank(),
              legend.text = element_text(size = 6.5)))
knitr::kable(data.frame(proxy = names(proxies),
                        RMSE_best_50pct = sapply(names(proxies), function(nm) cur$rmse[cur$proxy == nm][9]),
                        RMSE_all = sapply(names(proxies), function(nm) cur$rmse[cur$proxy == nm][19])),
             digits = 3, row.names = FALSE, caption = "RMSE of the most reliable half")
cat(sprintf("Spearman between |error| and the ensemble spread: %+.3f; tree SD ranges %.2f to %.2f\n",
            cor(err, sd_trees, method = "spearman"), min(sd_trees), max(sd_trees)))



<p class="caption"><b>Figure 3.</b> RMSE of the retained subset as a function of how much of the test set is kept, ranking compounds by three uncertainty proxies and by a random order as control; the dashed line is the experimental floor. A useful proxy bends the curve downwards on the left. Only the ensemble spread does: ranking by distance to the training set is no better than the random control.</p>

This figure corrects an impression the quartile table could leave. Ranking by **distance to the training set** does not improve the retained RMSE at all — the most reliable half by that criterion is no better than a random half — whereas ranking by **ensemble spread** cuts the RMSE of the retained half substantially. The two measurements are consistent: the RMSE is dominated by a minority of large errors (activity cliffs, mis-annotated records) that are not especially distant from the training set. The operational conclusion is uncomfortable for the classical applicability-domain literature and important for practice: **for per-compound triage, a model-internal difficulty estimate beats a chemical-distance domain**, while the distance remains worth reporting as a domain definition because it is computable for compounds that have never been measured.

<div class="exercise"><h4>Exercise 1</h4>
<p>Recompute the kNN-distance degradation table for the <em>scaffold</em>-split model, using its own training set as the reference (<code>knn_dist(TEs, TRs)</code>). Compare the four quartile means with those of the random split and explain why the nearest quartile of a scaffold split is already as distant as the third quartile of a random split. Then repeat the retention curve for that model: does the ensemble spread still win?</p></div>

## 4. Heuristic intervals, and whether they cover

Two intervals are routinely reported without being checked.

**Ensemble spread.** A random forest gives a prediction per tree; the standard deviation across trees is often quoted as an uncertainty, and \(\hat y \pm 1.645\,s\) as a 90 % interval<sup class="c"><a href="#r21">[21]</a></sup>. The hidden assumption is that the tree-to-tree spread equals the error distribution, which it does not: it measures disagreement among correlated models trained on the same data, not the irreducible noise.

**Quantile regression forests.** Instead of averaging the leaves, keep the whole conditional distribution of training responses in the leaves a compound falls into and read off its 5th and 95th percentiles<sup class="c"><a href="#r20">[20]</a></sup>; `ranger(quantreg = TRUE)` implements it. This at least targets quantiles of the conditional distribution, but the quantiles are estimated — with their own bias and over-fitting — and nothing guarantees that 90 % of future observations fall inside.

Both are tested below the only way that counts: by **empirical coverage**, the fraction of held-out compounds whose true value lies in the interval.



In [ ]:
cov_rf <- mean(abs(y[TE] - pred_te) <= 1.645*sd_trees)
qrf <- ranger(x = Xfp[TR, ], y = y[TR], num.trees = 300, min.node.size = 5,
              quantreg = TRUE, seed = 1, num.threads = 4)
qq <- predict(qrf, data = Xfp[TE, ], type = "quantiles", quantiles = c(0.05, 0.95))$predictions
cov_q <- mean(y[TE] >= qq[, 1] & y[TE] <= qq[, 2])
cat(sprintf("random forest +- 1.645 SD across trees : coverage %.3f (nominal 0.90), mean width %.3f log units\n",
            cov_rf, mean(2*1.645*sd_trees)))
cat(sprintf("quantile regression forest, 5th-95th   : coverage %.3f (nominal 0.90), mean width %.3f log units\n",
            cov_q, mean(qq[, 2] - qq[, 1])))
cat(sprintf("the tree-to-tree SD correlates with the absolute error at Spearman %+.3f -> it ranks difficulty usefully, but its scale is a guess\n",
            cor(abs(y[TE] - pred_te), sd_trees, method = "spearman")))



Neither heuristic lands on its nominal level: the ensemble interval under-covers and the quantile forest over-covers, each by a few points, and neither error is predictable in advance — which is the real objection, because an interval whose coverage you have to measure afterwards is not a guarantee. Notice the last statistic, though: the tree-to-tree spread *ranks* compounds by difficulty with a clear positive correlation to the actual error. That combination (useful ranking, wrong scale) is exactly what conformal prediction turns into a guarantee.

## 5. Inductive conformal prediction, written by hand

The recipe<sup class="c"><a href="#r8">[8]</a><a href="#r9">[9]</a></sup> has three steps and no model assumptions.

1. **Fit** the model on the training set (the *proper training set*).
2. **Calibrate**: on a held-out calibration set of size \(n\), compute a *nonconformity score* for every compound — for regression, the absolute residual \(\alpha_i = |y_i - \hat y_i|\).
3. **Predict**: for a confidence level \(1-\varepsilon\), take the \(k\)-th smallest calibration score with \[ k = \left\lceil (n+1)(1-\varepsilon) \right\rceil, \] and output the interval \(\hat y \pm \alpha_{(k)}\).

The guarantee is distribution-free: **if the calibration and test compounds are exchangeable, the interval contains the truth with probability at least \(1-\varepsilon\).** The proof is a rank argument in one line — under exchangeability the rank of a new compound's nonconformity score among the \(n+1\) scores is uniform, so the probability that it exceeds the \(k\)-th smallest is at most \(\varepsilon\). Everything about the model — bias, over-fitting, the shape of the residuals — is absorbed by the calibration scores.

### 5.1 Worked example by hand

Suppose a calibration set of \(n = 9\) compounds with absolute residuals, sorted:

| rank \(i\) | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|
| \(\alpha_{(i)}\) | 0.05 | 0.11 | 0.18 | 0.26 | 0.33 | 0.47 | 0.52 | 0.80 | 1.40 |

For \(\varepsilon = 0.2\) (80 % confidence): \(k = \lceil 10 \times 0.8 \rceil = 8\), so the half-width is \(\alpha_{(8)} = 0.80\). For \(\varepsilon = 0.1\): \(k = \lceil 10 \times 0.9 \rceil = 9\), half-width 1.40 — the largest calibration residual. For \(\varepsilon = 0.05\): \(k = \lceil 10 \times 0.95 \rceil = 10 > n\), and **no finite interval can be issued** from nine calibration points. That last line is the practical lesson: the calibration set must satisfy \(n \ge 1/\varepsilon - 1\).



In [ ]:
conformal_quantile <- function(scores, eps) {     # k-th smallest score, k = ceil((n+1)(1-eps))
  s <- sort(scores); n <- length(s); k <- ceiling((n + 1)*(1 - eps))
  if (k > n) NULL else s[k] }
demo <- c(0.05, 0.11, 0.18, 0.26, 0.33, 0.47, 0.52, 0.80, 1.40)
for (eps in c(0.2, 0.1, 0.05)) {
  k <- ceiling((length(demo) + 1)*(1 - eps)); q <- conformal_quantile(demo, eps)
  cat(sprintf("eps = %.2f: k = ceil(10 x %.2f) = %2d  ->  %s\n", eps, 1 - eps, k,
      if (is.null(q)) "k > n, no finite interval" else sprintf("half-width alpha_(%d) = %.2f", k, q))) }
cat(sprintf("minimum calibration size for 95%% confidence: n >= %d\n", ceiling(1/0.05) - 1))



### 5.2 The real calibration set



In [ ]:
sdtrees <- function(m, idx) apply(predict(m, data = Xfp[idx, ], predict.all = TRUE)$predictions, 1, sd)
icp <- function(m, ca, te, levels = c(.8, .9, .95), normalise = FALSE, beta = 0.3) {
  pc <- pr(m, ca); pt <- pr(m, te)
  if (normalise) { sc <- sdtrees(m, ca) + beta; st <- sdtrees(m, te) + beta
  } else          { sc <- rep(1, length(ca));   st <- rep(1, length(te)) }
  a <- abs(y[ca] - pc)/sc
  do.call(rbind, lapply(levels, function(L) {
    q <- conformal_quantile(a, 1 - L); half <- q*st
    data.frame(level = L, q = q, coverage = mean(abs(y[te] - pt) <= half),
               mean_width = mean(2*half), median_width = median(2*half),
               width_IQR = sprintf("%.2f-%.2f", quantile(2*half, .25), quantile(2*half, .75))) })) }
icp_plain <- icp(m_rand, CA, TE); icp_norm <- icp(m_rand, CA, TE, normalise = TRUE)
knitr::kable(icp_plain, digits = 3, row.names = FALSE,
             caption = "Plain inductive conformal prediction (random split)")
knitr::kable(icp_norm, digits = 3, row.names = FALSE,
             caption = "Difficulty-normalised (nonconformity divided by the ensemble spread + 0.3)")


In [ ]:
q90n <- conformal_quantile(abs(y[CA] - pr(m_rand, CA))/(sdtrees(m_rand, CA) + 0.3), 0.10)
st_te <- sdtrees(m_rand, TE) + 0.3
sel <- order(pred_te)[seq(1, length(TE), by = max(length(TE) %/% 60, 1))][1:60]
d4 <- data.frame(i = seq_along(sel), pred = pred_te[sel], obs = y[TE][sel], half = q90n*st_te[sel])
d4$miss <- abs(d4$obs - d4$pred) > d4$half
print(ggplot(d4, aes(i, pred)) +
        geom_errorbar(aes(ymin = pred - half, ymax = pred + half), colour = "grey75", width = 0, linewidth = 1.6) +
        geom_point(size = .8, colour = PAL["blue"]) +
        geom_point(aes(y = obs, colour = miss, shape = miss), size = 1.4) +
        scale_colour_manual(values = unname(PAL[c("green", "orange")]), guide = "none") +
        scale_shape_manual(values = c(1, 4), guide = "none") +
        labs(x = "test compounds, sorted by prediction", y = "pActivity",
             title = "90% normalised conformal intervals, one compound at a time"))
cat(sprintf("in this displayed subset of %d compounds, %d fall outside their 90%% interval (%.1f%%); over the whole test set %.1f%%\n",
            nrow(d4), sum(d4$miss), 100*mean(d4$miss),
            100*(1 - mean(abs(y[TE] - pred_te) <= q90n*st_te))))
cat(sprintf("interval half-widths in the subset: %.2f to %.2f log units\n", min(d4$half), max(d4$half)))



<p class="caption"><b>Figure 4.</b> Sixty test compounds sorted by predicted potency, with their 90 % difficulty-normalised conformal intervals (grey bars), the prediction (blue dot) and the measured value (green circle inside the interval, orange cross outside). Interval half-widths vary from 0.57 to 2.05 log units — more than a factor of three — and the misses are counted in the printed line.</p>

Both variants cover close to their nominal levels on this split, and they differ in *efficiency*: the normalised version produces intervals of similar average width but a wide interquartile range, because it gives narrow intervals to easy compounds and wide ones to hard ones. Validity is a property of the method; efficiency is what makes it useful<sup class="c"><a href="#r11">[11]</a></sup>. An interval that is always 2.5 log units wide is valid and almost useless for prioritisation; one that is 2.1 units wide for a quarter of the library and 2.8 for another quarter tells a chemist which predictions to act on.

A single split is one realisation of a random experiment, and the guarantee is on average over calibration sets. The next chunk therefore repeats the whole procedure with twenty different random partitions, and the one after asks how large a calibration set has to be.



In [ ]:
rows <- list()
for (seed in 0:19) {
  s2 <- random_split(nrow(Xd), seed = seed)
  m <- ranger(x = Xfp[s2$tr, ], y = y[s2$tr], num.trees = 200, min.node.size = 2,
              seed = 1, num.threads = 4)
  a  <- abs(y[s2$ca] - predict(m, data = Xfp[s2$ca, ])$predictions)
  pt <- predict(m, data = Xfp[s2$te, ])$predictions
  for (L in c(.8, .9, .95)) rows[[length(rows) + 1]] <-
    data.frame(seed = seed, level = L,
               coverage = mean(abs(y[s2$te] - pt) <= conformal_quantile(a, 1 - L))) }
RC <- do.call(rbind, rows)
agg <- do.call(data.frame, aggregate(coverage ~ level, RC,
         function(v) c(mean = mean(v), sd = sd(v), min = min(v), max = max(v))))
names(agg) <- c("level", "mean_coverage", "sd", "min", "max")
knitr::kable(agg, digits = 4, row.names = FALSE, caption = "Coverage over 20 independent random splits")
cat(sprintf("nominal levels are recovered on average to within %.3f; the spread across splits (SD %.3f to %.3f) is the sampling noise of a %d-compound test set\n",
            max(abs(agg$mean_coverage - agg$level)), min(agg$sd), max(agg$sd), length(TE)))


In [ ]:
csz <- do.call(rbind, lapply(c(20, 50, 100, 200, 400, length(CA)), function(nc) {
  co <- wi <- numeric(0)
  for (s in 1:15) { set.seed(s); sub <- sample(CA, nc)
    q <- conformal_quantile(abs(y[sub] - pr(m_rand, sub)), 0.10); if (is.null(q)) next
    co <- c(co, mean(abs(y[TE] - pred_te) <= q)); wi <- c(wi, 2*q) }
  data.frame(calibration_size = nc, mean_coverage = mean(co), sd_coverage = sd(co),
             mean_width = mean(wi), sd_width = sd(wi)) }))
knitr::kable(csz, digits = 3, row.names = FALSE, caption = "How large must the calibration set be?")
g1 <- ggplot(csz, aes(calibration_size, mean_coverage)) +
  geom_errorbar(aes(ymin = mean_coverage - sd_coverage, ymax = mean_coverage + sd_coverage), width = .05) +
  geom_line(colour = PAL["blue"]) + geom_point(colour = PAL["blue"]) +
  geom_hline(yintercept = 0.90, linetype = 2, colour = PAL["orange"]) + scale_x_log10() +
  labs(x = "calibration-set size", y = "empirical coverage at 90%", title = "Validity needs calibration data")
g2 <- ggplot(csz, aes(calibration_size, mean_width)) +
  geom_errorbar(aes(ymin = mean_width - sd_width, ymax = mean_width + sd_width), width = .05) +
  geom_line(colour = PAL["green"]) + geom_point(colour = PAL["green"]) + scale_x_log10() +
  labs(x = "calibration-set size", y = "90% interval width (log units)", title = "and so does efficiency")
print(g1 | g2)



<p class="caption"><b>Figure 5.</b> Coverage (left) and width (right) of the 90 % conformal interval as the calibration set is subsampled, each point averaged over 15 draws with its standard deviation. With a few dozen calibration compounds the coverage is roughly unbiased but wildly variable and the widths are erratic; stability arrives in the low hundreds. This is the practical reason to reserve a proper calibration split rather than reusing the test set.</p>

## 6. When the guarantee breaks: exchangeability and the scaffold split

Conformal validity requires only that the calibration and test compounds be **exchangeable** — that their joint distribution not depend on which is labelled "calibration". A random split satisfies it by construction. A scaffold split does not: the test compounds belong to chemical families chosen to be absent from the other parts<sup class="c"><a href="#r12">[12]</a><a href="#r13">[13]</a></sup>. The practical question is how much coverage is lost, and the realistic deployment scenario is worse than the symmetric one — in practice a model is calibrated on whatever data is at hand (same chemotypes as the training set) and then applied to new chemistry.



In [ ]:
pool <- c(TRs, CAs); set.seed(1); rp <- sample(length(pool))
TRm <- pool[rp[1:floor(.75*length(pool))]]; CAm <- pool[rp[(floor(.75*length(pool)) + 1):length(pool)]]
m_mix <- fit_rf(TRm)
scen <- list("A. random split (exchangeable)" = list(m_rand, CA, TE),
             "B. scaffold split, calibration also on new scaffolds" = list(m_scaf, CAs, TEs),
             "C. calibrated in-domain, tested on new scaffolds" = list(m_mix, CAm, TEs))
cover <- do.call(rbind, lapply(names(scen), function(nm) {
  mm <- scen[[nm]][[1]]; ca <- scen[[nm]][[2]]; te <- scen[[nm]][[3]]
  a <- abs(y[ca] - pr(mm, ca)); pt <- pr(mm, te)
  r <- data.frame(scenario = nm, RMSE = sqrt(mean((y[te] - pt)^2)))
  for (L in c(.8, .9, .95)) { q <- conformal_quantile(a, 1 - L)
    r[[paste0("cov", L*100)]] <- mean(abs(y[te] - pt) <= q); r[[paste0("w", L*100)]] <- 2*q }
  r }))
knitr::kable(cover, digits = 3, row.names = FALSE, caption = "Three scenarios, same model family")
cat("coverage shortfall (empirical minus nominal):\n")
for (i in 1:3) cat(sprintf("  %-52s 80%%: %+.3f   90%%: %+.3f   95%%: %+.3f\n",
  substr(cover$scenario[i], 1, 52), cover$cov80[i] - .8, cover$cov90[i] - .9, cover$cov95[i] - .95))


In [ ]:
levs <- c(.5, .6, .7, .8, .85, .9, .95, .99)
a_r <- abs(y[CA] - pr(m_rand, CA)); a_s <- abs(y[CAs] - pr(m_scaf, CAs)); pt_s <- pr(m_scaf, TEs)
cc <- rbind(data.frame(level = levs, split = "random",
              cov = sapply(levs, function(L) mean(abs(y[TE] - pred_te) <= conformal_quantile(a_r, 1 - L)))),
            data.frame(level = levs, split = "scaffold",
              cov = sapply(levs, function(L) mean(abs(y[TEs] - pt_s) <= conformal_quantile(a_s, 1 - L)))))
g1 <- ggplot(cc, aes(level, cov, colour = split, shape = split)) +
  geom_abline(slope = 1, intercept = 0, linetype = 2) + geom_line() + geom_point() +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")])) +
  labs(x = "nominal level 1 - eps", y = "empirical coverage", title = "Coverage plot") +
  theme(legend.position = "bottom", legend.title = element_blank())
sh <- do.call(rbind, lapply(1:3, function(i) data.frame(
  scenario = c("A. random", "B. scaffold\n(cal. also new)", "C. in-domain cal.\n-> new scaffolds")[i],
  level = factor(c("80%", "90%", "95%")),
  shortfall = c(cover$cov80[i] - .8, cover$cov90[i] - .9, cover$cov95[i] - .95))))
g2 <- ggplot(sh, aes(scenario, shortfall, fill = level)) +
  geom_col(position = position_dodge(width = .8), width = .7) +
  geom_hline(yintercept = 0, linewidth = .3) + scale_fill_manual(values = unname(PAL)) +
  labs(x = NULL, y = "empirical coverage - nominal", title = "Where the conformal guarantee is lost") +
  theme(axis.text.x = element_text(size = 6.5), legend.position = "bottom", legend.title = element_blank())
print(g1 | g2)



<p class="caption"><b>Figure 6.</b> Left: empirical coverage against nominal level under the random split (blue) and the scaffold split (orange); the diagonal is perfect validity. The random curve tracks the diagonal; the scaffold curve falls below it, and the gap widens at high confidence, where the guarantee is needed most. Right: coverage minus nominal for the three scenarios at three levels — scenario A scatters around zero, both scaffold scenarios fall below it, and scenario C is short at every level while also issuing the narrowest intervals of the three.</p>

Read the shortfalls, not the coverages. Under the random split the predictor is valid to within sampling noise. Under both scaffold scenarios the coverage falls short — and in scenario C, the realistic one, the shortfall is largest *and* the intervals are *narrower* than in scenario B, which is the dangerous combination: confident and wrong. The calibration residuals in C were collected on compounds the model finds easy, so the quantile they produce is too small for the compounds it is applied to.

Three consequences for practice:

1. **Calibrate on data that resembles the deployment population.** If the model will be applied to a new chemotype, the calibration set must contain new chemotypes — which is what scenario B does, and why its shortfall is smaller.
2. **Report coverage on a split that matches the claim.** A conformal predictor validated only on a random split carries no guarantee for new chemistry; Norinder and co-workers' introduction of conformal prediction to QSAR<sup class="c"><a href="#r14">[14]</a></sup> and the applied studies that followed<sup class="c"><a href="#r15">[15]</a><a href="#r16">[16]</a></sup> all make this the central practical caveat.
3. **Combine with the applicability domain.** Compute the distance, refuse or widen predictions outside it, and report conformal intervals for the rest with coverage measured on a comparable split<sup class="c"><a href="#r17">[17]</a></sup>.

## 7. Conformal classification: prediction sets instead of labels

For the potency-class problem (pActivity ≥ 7 or not) the same recipe produces a **prediction set** rather than a label. Using \(\alpha = 1 - \hat p(\text{class})\) as the nonconformity score, a class is included when its nonconformity does not exceed the calibration quantile. Four outcomes are possible: a singleton (confident), both classes (uninformative but honest), the empty set (the compound conforms to neither class — a flag for an outlier), and, for a singleton, the possibility of being wrong at the nominal rate.

The **Mondrian** (class-conditional) variant calibrates a separate quantile per class, which matters whenever the classes are imbalanced: plain conformal classification guarantees *marginal* coverage and can achieve it by covering the majority class very well and the minority class badly.



In [ ]:
clf <- ranger(x = Xfp[TR, ], y = factor(ycls[TR]), num.trees = 300, min.node.size = 2,
              probability = TRUE, seed = 1, num.threads = 4)
Pca <- predict(clf, data = Xfp[CA, ])$predictions; Pte <- predict(clf, data = Xfp[TE, ])$predictions
conformal_sets <- function(eps, mondrian = TRUE) {
  inc <- matrix(FALSE, length(TE), 2)
  for (k in 0:1) {
    a <- if (mondrian) 1 - Pca[ycls[CA] == k, k + 1]              # calibrate within the class
         else          1 - Pca[cbind(seq_along(CA), ycls[CA] + 1)]  # one pooled quantile
    inc[, k + 1] <- (1 - Pte[, k + 1]) <= conformal_quantile(a, eps) }
  inc }
cc2 <- do.call(rbind, lapply(c(.2, .1, .05), function(eps) do.call(rbind, lapply(c(FALSE, TRUE), function(mo) {
  inc <- conformal_sets(eps, mo); hit <- inc[cbind(seq_along(TE), ycls[TE] + 1)]
  data.frame(level = 1 - eps, variant = ifelse(mo, "Mondrian", "plain"), coverage = mean(hit),
             coverage_class0 = mean(hit[ycls[TE] == 0]), coverage_class1 = mean(hit[ycls[TE] == 1]),
             singletons = mean(rowSums(inc) == 1), both_classes = mean(rowSums(inc) == 2),
             empty = mean(rowSums(inc) == 0)) }))))
knitr::kable(cc2, digits = 3, row.names = FALSE, caption = "Conformal classification: plain vs Mondrian")
cat(sprintf("class prevalence in the test set: %.3f potent / %.3f not\n",
            mean(ycls[TE]), 1 - mean(ycls[TE])))


In [ ]:
cl <- do.call(rbind, lapply(1:nrow(cc2), function(i) data.frame(
  level = sprintf("%.0f%%", 100*cc2$level[i]), variant = cc2$variant[i],
  class = c("class 0 (not potent)", "class 1 (potent)"),
  cov = c(cc2$coverage_class0[i], cc2$coverage_class1[i]))))
g1 <- ggplot(cl, aes(level, cov, fill = interaction(variant, class, sep = ", "))) +
  geom_col(position = position_dodge(width = .85), width = .8) +
  geom_point(data = data.frame(level = c("80%", "90%", "95%"), cov = c(.8, .9, .95)),
             aes(level, cov), inherit.aes = FALSE, shape = 95, size = 8) +
  coord_cartesian(ylim = c(0.5, 1.02)) + scale_fill_manual(values = unname(PAL)) +
  labs(x = NULL, y = "coverage within the class", title = "Marginal validity can hide class imbalance") +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 5.6))
mo <- subset(cc2, variant == "Mondrian")
comp <- do.call(rbind, lapply(1:3, function(i) data.frame(
  level = sprintf("%.0f%%", 100*mo$level[i]),
  kind = factor(c("singleton", "both classes", "empty"), levels = c("singleton", "both classes", "empty")),
  frac = c(mo$singletons[i], mo$both_classes[i], mo$empty[i]))))
g2 <- ggplot(comp, aes(level, frac, fill = kind)) + geom_col(width = .7) +
  scale_fill_manual(values = unname(PAL[c("green", "blue", "orange")])) +
  labs(x = NULL, y = "fraction of test compounds", title = "Set composition (Mondrian)") +
  theme(legend.position = "bottom", legend.title = element_blank())
print(g1 | g2)



<p class="caption"><b>Figure 7.</b> Left: coverage within each class for the plain and Mondrian conformal classifiers at three confidence levels, with the nominal level marked by a dash. The plain variant reaches its marginal target by over-covering the majority class and under-covering the minority one — most visibly at 80 %, where class 1 is covered 0.70 against the nominal 0.80 — while the Mondrian variant holds both classes near target. Right: composition of the Mondrian prediction sets; as confidence rises, the fraction of informative singletons falls and "both classes" grows.</p>

The right-hand panel is the honest trade: a 95 % guarantee on a model with this discrimination can only be bought by refusing to decide for a large fraction of compounds. That is not a defect, it is information — it tells a project how many compounds can actually be triaged at the confidence it requires. The empty sets at 80 % are the other half of the story: compounds that conform to neither class and should be treated as outside the domain rather than predicted.

## 8. Probability calibration

A prediction set answers "which classes are plausible". A calibrated probability answers "how often is a compound with score 0.7 actually potent" — the quantity a decision needs when costs are asymmetric. Two standard recalibrations exist: **Platt scaling**, a logistic fit of the label on the score<sup class="c"><a href="#r23">[23]</a></sup>, and **isotonic regression**, a monotone step function<sup class="c"><a href="#r22">[22]</a></sup> — `stats::isoreg` in base R — both fitted on the calibration set, never on the training or test data. Quality is read from a **reliability diagram** and summarised by the **Brier score**<sup class="c"><a href="#r24">[24]</a></sup>, the mean squared difference between probability and outcome<sup class="c"><a href="#r25">[25]</a></sup>.



In [ ]:
p_raw <- Pte[, 2]; p_cal <- Pca[, 2]
platt <- glm(ycls[CA] ~ p_cal, family = binomial)
p_platt <- as.numeric(predict(platt, newdata = data.frame(p_cal = p_raw), type = "response"))
o <- order(p_cal); iso <- isoreg(p_cal[o], ycls[CA][o])
p_iso <- pmin(pmax(approxfun(iso$x, iso$yf, rule = 2, ties = "ordered")(p_raw), 0), 1)
brier <- function(p) mean((p - ycls[TE])^2)
auc <- function(lab, sc) { r <- rank(sc); n1 <- sum(lab == 1); n0 <- sum(lab == 0)
                           (sum(r[lab == 1]) - n1*(n1 + 1)/2)/(n1*n0) }
calcurve <- function(p, nb = 8) {
  b <- cut(p, unique(quantile(p, 0:nb/nb)), include.lowest = TRUE)
  data.frame(mean_p = tapply(p, b, mean), obs = tapply(ycls[TE], b, mean)) }
cal <- do.call(rbind, lapply(c("raw random forest", "Platt (sigmoid)", "isotonic"), function(nm) {
  p <- switch(nm, "raw random forest" = p_raw, "Platt (sigmoid)" = p_platt, p_iso)
  cv <- calcurve(p)
  data.frame(probabilities = nm, Brier = brier(p), ROC_AUC = auc(ycls[TE], p),
             mean_predicted = mean(p), observed_prevalence = mean(ycls[TE]),
             calibration_error = mean(abs(cv$obs - cv$mean_p), na.rm = TRUE)) }))
knitr::kable(cal, digits = 4, row.names = FALSE, caption = "Three sets of probabilities for the same model")
cat(sprintf("Brier score of always predicting the prevalence (%.3f): %.4f\n",
            mean(ycls[TE]), mean((mean(ycls[TE]) - ycls[TE])^2)))


In [ ]:
rel <- do.call(rbind, lapply(c("raw RF", "Platt", "isotonic"), function(nm) {
  p <- switch(nm, "raw RF" = p_raw, "Platt" = p_platt, p_iso)
  cv <- calcurve(p); data.frame(method = nm, mean_p = cv$mean_p, obs = cv$obs) }))
g1 <- ggplot(rel, aes(mean_p, obs, colour = method, shape = method)) +
  geom_abline(slope = 1, intercept = 0, linetype = 2) + geom_line() + geom_point() +
  scale_colour_manual(values = unname(PAL)) +
  labs(x = "mean predicted probability", y = "observed fraction potent", title = "Reliability diagram") +
  theme(legend.position = "bottom", legend.title = element_blank())
g2 <- ggplot(data.frame(p = p_raw, cls = factor(ycls[TE], 0:1, c("not potent", "potent"))),
             aes(p, fill = cls)) +
  geom_histogram(bins = 25, alpha = .6, position = "identity") +
  scale_fill_manual(values = unname(PAL[c("blue", "orange")])) +
  labs(x = "predicted P(potent)", y = "test compounds", title = "Score distribution by true class") +
  theme(legend.position = "bottom", legend.title = element_blank())
print(g1 | g2)



<p class="caption"><b>Figure 8.</b> Left: reliability diagram with eight quantile bins — the raw forest probabilities already lie close to the diagonal, and neither Platt scaling nor isotonic regression improves the Brier score. Right: the distribution of raw scores by true class, which shows why: averaging over 300 trees produces scores that are neither saturated at 0/1 nor compressed around the prevalence.</p>

The negative result is worth keeping: **recalibration is not always needed, and you only know by measuring.** Random forests with many trees are often close to calibrated on balanced-ish problems, whereas boosted trees and neural networks are typically over-confident. The decision rule is simple: fit Platt and isotonic on the calibration set, compare Brier scores and reliability curves on the test set, and keep the raw probabilities if they win — as they do here.

<div class="exercise"><h4>Exercise 2</h4>
<p>Replace the forest with a gradient-boosting classifier (<code>xgboost</code> with <code>objective = "binary:logistic"</code>, 400 rounds) and repeat the comparison. Boosted trees are usually over-confident: report the Brier score and the calibration error before and after Platt scaling, and say which of the two recalibrations you would deploy and why. Then recompute the conformal prediction sets of Section 7 with the new classifier — does the <em>coverage</em> change, and should it?</p></div>

## 9. The OECD principles and a QMRF-style report

Regulatory and editorial expectations for a (Q)SAR model are codified in the five **OECD principles**: a model submitted for a regulatory purpose should have (1) a defined endpoint, (2) an unambiguous algorithm, (3) a defined domain of applicability, (4) appropriate measures of goodness-of-fit, robustness and predictivity, and (5) a mechanistic interpretation, if possible. The reporting vehicle is the **QSAR Model Reporting Format (QMRF)**; Tropsha's best-practice paper<sup class="c"><a href="#r18">[18]</a></sup> and Tetko and co-workers' critical assessment<sup class="c"><a href="#r19">[19]</a></sup> are the practical commentaries to read alongside the guidance.

Everything the first four principles require has been computed here, so the report can be generated rather than written. Principle 5 is the one that cannot be automated.



In [ ]:
q90_plain <- conformal_quantile(abs(y[CA] - pr(m_rand, CA)), 0.10)
report <- sprintf('
QSAR MODEL REPORTING FORMAT (QMRF-style summary) - generated by this document
=============================================================================
1. QSAR identifier
   Title            : Random-forest regressor/classifier of ROCK1 inhibitory potency
   Software         : R %s, ranger %s, glmnet %s; forest seed 1

2. General information
   Endpoint         : pActivity = -log10(IC50/Ki/Kd in M) against human ROCK1 (ChEMBL CHEMBL3231)
   Data source      : ChEMBL release 37, curated table rock1_chembl_curated.csv
   Compounds        : %d standardised parent structures with at least one exact measurement
   Experimental unc.: pooled within-compound SD = %.3f log units (%d replicated compounds)

3. Defined endpoint (principle 1)
   Regression       : pActivity (continuous, %.2f to %.2f)
   Classification   : potent if pActivity >= 7.0 (100 nM); prevalence %.3f

4. Unambiguous algorithm (principle 2)
   Descriptors      : Morgan fingerprint, radius 2, 2048 bits (frozen file; see data/manifest_fingerprints.json)
   Algorithm        : ranger random forest, 300 trees, min.node.size = 2, seed = 1
   Splits           : random 60/20/20 and deterministic Bemis-Murcko scaffold 60/20/20

5. Applicability domain (principle 3)
   Definition       : mean Tanimoto distance to the 5 nearest training compounds < %.2f
                      (threshold = 90th percentile of the calibration set)
   Coverage         : %.1f%% of random-split test compounds inside
   Evidence         : MAE %.3f (nearest quartile) -> %.3f (farthest quartile); errors > 1 log unit
                      %.1f%% -> %.1f%%. NOTE: the relationship is not monotone and the retention
                      experiment shows this domain ranks individual compounds no better than chance.
   Secondary        : descriptor bounding box (%.1f%% inside), leverage h* = %.4f for the ridge reference

6. Goodness-of-fit, robustness and predictivity (principle 4)
   Random split     : R2 %.3f, MAE %.3f, RMSE %.3f log units
   Scaffold split   : R2 %.3f, MAE %.3f, RMSE %.3f log units
   Classification   : ROC-AUC %.3f, Brier %.4f (raw forest probabilities, no recalibration)
   Uncertainty      : inductive conformal, 90%% interval width %.2f log units, empirical coverage %.3f
                      (random split); %.3f when calibrated in-domain and tested on new scaffolds
   Benchmark        : experimental floor RMSE >= %.3f; model/floor ratio %.2f

7. Mechanistic interpretation (principle 5)
   Not established automatically. Fingerprint-bit attributions are consistent with the known
   hinge-binding motif of ROCK inhibitors but are confounded with training-set frequency.

8. Limitations
   - Public data from heterogeneous assays; IC50, Ki and Kd pooled after standardisation.
   - %d censored compounds excluded from the regression target.
   - Conformal guarantees hold under exchangeability; coverage degrades on new scaffolds (section 6).
',
paste(R.version$major, R.version$minor, sep = "."), as.character(packageVersion("ranger")),
as.character(packageVersion("glmnet")), nrow(cmp), pooled_sd, nrow(rep_), min(y), max(y), mean(ycls),
thr_knn, 100*mean(d_knn < thr_knn), deg$MAE[1], deg$MAE[4],
100*deg$frac_error_gt_1[1], 100*deg$frac_error_gt_1[4], 100*mean(in_box), h_star,
info$R2[1], info$MAE[1], info$RMSE[1], info$R2[2], info$MAE[2], info$RMSE[2],
auc(ycls[TE], p_raw), brier(p_raw), 2*q90_plain, icp_plain$coverage[2], cover$cov90[3],
pooled_sd, info$RMSE[1]/pooled_sd, sum(df$censored))
cat(report)



<div class="box bridge"><h4>Bridge</h4>
<p>Regulatory QSAR vocabulary, descriptors and the ADMET context are developed in the cheminformatics course: <a href="../../../Cheminformatics/U12_Classical_QSAR/index.html">U12 (classical QSAR)</a> and <a href="../../../Cheminformatics/U14_Validation/index.html">U14 (validation)</a>, with the chemical-space view in <a href="../../../Cheminformatics/U8_Chemical_Space/index.html">U8</a>. Protein-structure confidence scores — pLDDT and PAE, the structural analogue of everything in this unit — appear in the bioinformatics course, <a href="../../../BGM324/02_Protein_Structure_Prediction/index.html">BGM324 unit 02</a>, and docking-score reliability in <a href="../../../BGM324/03_Molecular_Docking/index.html">unit 03</a>. The probabilistic background is the artificial-intelligence course, <a href="../../../BGM411/R/index.html">BGM411 (R edition)</a>. Inside this course, <a href="../U15_Clustering/index.html">Unit 15</a> built the Tanimoto machinery the applicability domain uses, <a href="../U16_Interpretability/index.html">Unit 16</a> asked why a prediction came out as it did, and <a href="../U18_Deep_Learning/index.html">Unit 18</a> asks whether a neural network would do better. The Python edition of this unit is <a href="../../Python/U17_Uncertainty_AD/index.html">here</a>.</p></div>



In [ ]:
final <- data.frame(
  quantity = c("experimental floor (pooled replicate SD)", "model RMSE, random split",
               "model RMSE, scaffold split", "AD: bounding box", "AD: leverage (ridge model)",
               "AD: kNN Tanimoto distance", "RF +-1.645 SD interval", "quantile regression forest",
               "conformal, random split", "conformal, 20 repetitions",
               "conformal, in-domain calibration -> new scaffolds",
               "conformal classification (Mondrian, 90%)", "probability calibration"),
  value = c(sprintf("%.3f log units", pooled_sd), sprintf("%.3f", info$RMSE[1]),
            sprintf("%.3f", info$RMSE[2]), sprintf("%.1f%% inside", 100*mean(in_box)),
            sprintf("%.1f%% outside h*", 100*mean(out_lev)), sprintf("median %.3f", median(d_knn)),
            sprintf("coverage %.3f", cov_rf), sprintf("coverage %.3f", cov_q),
            sprintf("coverage %.3f", icp_plain$coverage[2]),
            sprintf("mean coverage %.3f", agg$mean_coverage[2]),
            sprintf("coverage %.3f", cover$cov90[3]),
            sprintf("singletons %.2f", cc2$singletons[cc2$variant == "Mondrian" & cc2$level == 0.90]),
            sprintf("Brier raw %.4f", brier(p_raw))),
  reading = c("aleatoric; irreducible", sprintf("%.2f x the floor", info$RMSE[1]/pooled_sd),
              sprintf("%.2f x the floor", info$RMSE[2]/pooled_sd), "no useful degradation outside",
              sprintf("MAE %.2f -> %.2f", mean(abs(res[!out_lev])), mean(abs(res[out_lev]))),
              sprintf("MAE %.2f -> %.2f across quartiles, non-monotone", deg$MAE[1], deg$MAE[4]),
              "nominal 0.90", "nominal 0.90",
              sprintf("width %.2f log units", icp_plain$mean_width[2]),
              sprintf("SD across splits %.3f", agg$sd[2]), "guarantee lost: exchangeability fails",
              "the price of a guarantee is abstention",
              sprintf("Platt %.4f, isotonic %.4f", brier(p_platt), brier(p_iso))))
knitr::kable(final, row.names = FALSE)



<p class="caption"><b>Table 1.</b> Every uncertainty quantity computed in this unit. Four rows are negative results — the bounding box does not work, the distance domain does not rank compounds, and neither heuristic interval hits its nominal level.</p>

## 10. Summary

**Two uncertainties, one floor.** Aleatoric uncertainty is noise in the measurement and cannot be modelled away; epistemic uncertainty is distance from what the model has seen and can be. The dataset's own replicates put the aleatoric floor at about half a log unit<sup class="c"><a href="#r2">[2]</a><a href="#r3">[3]</a></sup>, so a random-split RMSE of a small multiple of that is close to the best achievable, and every error analysis should be read against it.

**Applicability domain.** Four definitions were computed and judged by whether accuracy degrades outside them<sup class="c"><a href="#r4">[4]</a></sup>. The descriptor bounding box classified almost everything as inside. Leverage with the \(h^{*}=3p/n\) threshold and the Williams plot worked for the model it is defined for — the descriptor-space linear model<sup class="c"><a href="#r6">[6]</a></sup> — and says nothing about a fingerprint forest. Distance to the nearest training neighbours, computed in the model's own representation, separated the extreme quartiles only weakly and **not monotonically**: the nearest quartile was as bad as the farthest, because that is where activity cliffs live. In the retention experiment it ranked individual compounds no better than chance, while the model's own ensemble spread cut the RMSE of the retained half substantially. The rule is to define the domain in the model's space, publish the degradation table including its non-monotonicity, and use a model-internal difficulty estimate for per-compound triage.

**Intervals.** Ensemble spread and quantile regression forests<sup class="c"><a href="#r20">[20]</a></sup> missed the nominal 90 % in opposite directions. Inductive conformal prediction<sup class="c"><a href="#r8">[8]</a><a href="#r9">[9]</a><a href="#r10">[10]</a></sup> — three steps, a dozen lines, no distributional assumptions — reached its nominal coverage on a random split and on average over twenty repetitions, and the difficulty-normalised variant redistributed the same average width towards the compounds the model finds hard. Validity is guaranteed; efficiency is what you optimise. Both need calibration data: with twenty calibration compounds the coverage standard deviation across draws was several times larger than with four hundred.

**Where the guarantee goes.** Exchangeability is the only assumption, and a scaffold split violates it. Coverage fell short under both scaffold scenarios and most dangerously in the realistic one — calibrated on familiar chemistry, applied to new chemotypes — where the intervals were simultaneously narrower and less reliable<sup class="c"><a href="#r12">[12]</a><a href="#r13">[13]</a></sup>.

**Sets and probabilities.** Conformal classification returns prediction sets whose composition makes the cost of confidence explicit, and the Mondrian variant restores the per-class validity that the marginal guarantee hides. Probability recalibration is cheap, must be fitted on a held-out calibration set, and on this random forest was not needed — a conclusion that required measuring the Brier score and the reliability curve<sup class="c"><a href="#r24">[24]</a></sup>.

**Reporting.** The five OECD principles map onto the sections of this unit, and the QMRF-style report was generated from the computed quantities, with the limitations section carrying the censored compounds, the assay heterogeneity and the coverage loss on new scaffolds. All of it runs in base R<sup class="c"><a href="#r30">[30]</a></sup> with `ranger`<sup class="c"><a href="#r31">[31]</a></sup>, `glmnet`<sup class="c"><a href="#r32">[32]</a></sup> and `ggplot2`<sup class="c"><a href="#r33">[33]</a></sup>.

## 11. Self-assessment

<details><summary>1. Separate the two uncertainties for a concrete case: a model predicts pActivity 7.2 for a compound from a chemotype absent from the training set, measured once in a single laboratory.</summary>
<p>Aleatoric: the single measurement carries the assay's own error, about half a log unit for public data of this kind (pooled SD 0.506 here), which no model can remove. Epistemic: the compound is far from the training set in fingerprint space, so the model is extrapolating and its error is likely larger than its average test error. The first is reported as a floor; the second is detected by the applicability domain and absorbed into a wider conformal interval — provided the conformal predictor was calibrated on comparable chemistry.</p></details>

<details><summary>2. Why is a descriptor bounding box a poor applicability domain for a fingerprint model?</summary>
<p>Because it is measured in a space the model does not use, and because it is a union of one-dimensional constraints: a molecule can satisfy all eleven descriptor ranges and still occupy a region of chemical space with no training compounds in it. In Section 3 it declared about 98 % of test compounds inside and the few outside were barely predicted worse.</p></details>

<details><summary>3. What is h*, and what does a point in the top-right corner of a Williams plot mean?</summary>
<p>h* = 3p/n is the conventional leverage threshold for a linear model with p parameters and n training rows; leverage is the diagonal of the hat matrix, measuring distance from the centre of the training data in the model's metric. Top-right means high leverage and a large standardised residual: an influential compound that the model also predicts badly — the case that most deserves inspection before publication.</p></details>

<details><summary>4. Write down the conformal quantile index and explain the +1.</summary>
<p>k = ceil((n+1)(1−eps)) with n the calibration size. The +1 accounts for the test compound itself: validity comes from the rank of its nonconformity score among the n+1 scores being uniform under exchangeability. It also implies n ≥ 1/eps − 1, so 95 % confidence is impossible with fewer than 19 calibration points, as the worked example in Section 5.1 shows.</p></details>

<details><summary>5. A referee says "conformal prediction is distribution-free, so it needs no validation". Answer.</summary>
<p>It is free of assumptions about the data distribution and about the model, but not of the exchangeability assumption between calibration and test data. Section 6 showed coverage falling below nominal under a scaffold split — the deployment scenario a QSAR paper usually claims — so empirical coverage must be reported on the split that matches the claim, exactly like a point performance metric.</p></details>

<details><summary>6. Distinguish validity from efficiency, and give the numbers from this unit.</summary>
<p>Validity is whether the empirical coverage reaches the nominal level; efficiency is how narrow the intervals are. The plain predictor gave every compound the same 90 % width of about 2.5 log units; the difficulty-normalised predictor kept comparable coverage while varying widths across an interquartile range of roughly 2.1 to 2.8 log units, narrow for easy compounds and wide for hard ones.</p></details>

<details><summary>7. Why does an ensemble-spread interval miss its nominal level even though the spread correlates with the error?</summary>
<p>Because the spread measures disagreement among trees trained on the same data — it ignores the shared bias and the aleatoric noise, and the factor 1.645 assumes a normal error of exactly that scale. The correlation (positive, moderate) means it ranks difficulty well, which is why it makes a good <em>normaliser</em> inside a conformal predictor: conformal supplies the scale, the ensemble supplies the ranking.</p></details>

<details><summary>8. When does a conformal classifier return an empty set, and what should you do with that compound?</summary>
<p>When the predicted probability of every class is below the calibration threshold — the compound conforms to no class. In Section 7 this happened for a small percentage of compounds at the 80 % level and disappeared at higher confidence. It is a flag, not an answer: treat it as outside the applicability domain and either gather more data of that kind or decline to predict.</p></details>

<details><summary>9. What does Mondrian conformal prediction fix?</summary>
<p>Class-conditional validity. The plain version guarantees only marginal coverage and can reach it by covering the majority class far better than the minority: in Section 7 the plain variant's class-1 coverage fell to 0.70 at the 80 % level while class 0 was covered 0.86. Mondrian calibrates one quantile per class, keeping both near the target.</p></details>

<details><summary>10. Your random forest's Brier score does not improve after Platt scaling. What do you report?</summary>
<p>That the raw probabilities were already close to calibrated, with the reliability diagram and the Brier scores of all three variants as evidence, and that the raw model is therefore deployed. Recalibration is a hypothesis to test on held-out data, not a mandatory step; boosted trees and neural networks usually need it, forests with many trees often do not.</p></details>

<details><summary>11. Map the five OECD principles onto the sections of this unit.</summary>
<p>(1) Defined endpoint: the pActivity definition and the 7.0 class boundary of Section 2; (2) unambiguous algorithm: the fingerprint and forest specification, including the seed; (3) applicability domain: Section 3, with the degradation table <em>and</em> its failure as evidence; (4) goodness-of-fit, robustness and predictivity: the two splits, the conformal coverage and the experimental-floor comparison; (5) mechanistic interpretation: Unit 16, with its own controls — and the only principle that cannot be generated automatically.</p></details>

<details><summary>12. A colleague asks for the single most useful uncertainty number for prioritising 10,000 catalogue compounds. What do you give?</summary>
<p>Per-compound: a difficulty-normalised conformal interval at the confidence the project needs, with the ensemble spread as the normaliser — the distance to the training set alone would not rank them. Aggregate: the empirical coverage of that predictor measured on a split that resembles the catalogue — a scaffold or cluster split, not a random one — so that the number of compounds expected to be mis-prioritised can be stated in advance.</p></details>

## Glossary

<dl class="gloss">
<dt>Aleatoric uncertainty</dt><dd>Irreducible noise in the data-generating process, including measurement error; unaffected by more data.</dd>
<dt>Applicability domain (AD)</dt><dd>The region of chemical/feature space in which a model's predictions are considered reliable, with a stated definition and evidence of degradation outside it.</dd>
<dt>Brier score</dt><dd>Mean squared difference between predicted probability and binary outcome; lower is better.</dd>
<dt>Calibration set</dt><dd>Data held out from training and from testing, used to fit the uncertainty (conformal quantiles, Platt/isotonic maps).</dd>
<dt>Conformal prediction (inductive)</dt><dd>Distribution-free framework producing intervals or prediction sets with guaranteed marginal coverage under exchangeability, using nonconformity scores from a calibration set.</dd>
<dt>Coverage</dt><dd>Fraction of test cases whose true value lies inside the predicted interval or set; compared with the nominal level 1 − eps.</dd>
<dt>Efficiency</dt><dd>Narrowness of valid intervals or smallness of valid prediction sets.</dd>
<dt>Epistemic uncertainty</dt><dd>Uncertainty from the model's ignorance (extrapolation, limited data); reducible with relevant additional data.</dd>
<dt>Exchangeability</dt><dd>Invariance of a joint distribution to permutation of the observations; the only assumption behind conformal validity.</dd>
<dt>Isotonic regression (calibration)</dt><dd>Monotone step-function recalibration of scores to probabilities; <code>stats::isoreg</code> in base R.</dd>
<dt>Leverage / h*</dt><dd>Diagonal of the hat matrix measuring a point's distance from the training centroid in a linear model's metric; the 3p/n threshold marks structural outliers.</dd>
<dt>Mondrian conformal prediction</dt><dd>Class-conditional (or group-conditional) calibration, giving per-class rather than only marginal validity.</dd>
<dt>Nonconformity score</dt><dd>A measure of how unusual a (feature, label) pair is for the fitted model, e.g. the absolute residual or 1 − predicted class probability.</dd>
<dt>Platt scaling</dt><dd>Logistic recalibration of a score to a probability, fitted on held-out data.</dd>
<dt>Prediction set</dt><dd>The set of labels a conformal classifier cannot exclude at the chosen confidence; may be a singleton, several classes or empty.</dd>
<dt>QMRF</dt><dd>QSAR Model Reporting Format: the structured report whose sections follow the five OECD validation principles.</dd>
<dt>Quantile regression forest</dt><dd>Random forest that keeps the full conditional distribution in its leaves, so any quantile of the response can be read off.</dd>
<dt>Reliability diagram</dt><dd>Observed event frequency against mean predicted probability per bin; the diagonal is perfect calibration.</dd>
<dt>Retention curve</dt><dd>Error of the retained subset as a function of how much of the test set is kept, ranked by an uncertainty proxy.</dd>
<dt>Williams plot</dt><dd>Standardised residual against leverage, with h* and ±3 reference lines; the classical QSAR outlier diagnostic.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Hullermeier E, Waegeman W. Aleatoric and epistemic uncertainty in machine learning: an introduction to concepts and methods. <em>Mach Learn</em>. 2021;110:457-506. <a href="https://doi.org/10.1007/s10994-021-05946-3">doi:10.1007/s10994-021-05946-3</a></li>
<li id="r2">Kramer C, Kalliokoski T, Gedeck P, Vulpetti A. The experimental uncertainty of heterogeneous public Ki data. <em>J Med Chem</em>. 2012;55(11):5165-5173. <a href="https://doi.org/10.1021/jm300131x">doi:10.1021/jm300131x</a></li>
<li id="r3">Kalliokoski T, Kramer C, Vulpetti A, Gedeck P. Comparability of mixed IC50 data - a statistical analysis. <em>PLoS ONE</em>. 2013;8(4):e61007. <a href="https://doi.org/10.1371/journal.pone.0061007">doi:10.1371/journal.pone.0061007</a></li>
<li id="r4">Sahigara F, Mansouri K, Ballabio D, Mauri A, Consonni V, Todeschini R. Comparison of different approaches to define the applicability domain of QSAR models. <em>Molecules</em>. 2012;17(5):4791-4810. <a href="https://doi.org/10.3390/molecules17054791">doi:10.3390/molecules17054791</a></li>
<li id="r5">Jaworska J, Nikolova-Jeliazkova N, Aldenberg T. QSAR applicability domain estimation by projection of the training set in descriptor space: a review. <em>Altern Lab Anim</em>. 2005;33(5):445-459. <a href="https://doi.org/10.1177/026119290503300508">doi:10.1177/026119290503300508</a></li>
<li id="r6">Gramatica P. Principles of QSAR models validation: internal and external. <em>QSAR Comb Sci</em>. 2007;26(5):694-701. <a href="https://doi.org/10.1002/qsar.200610151">doi:10.1002/qsar.200610151</a></li>
<li id="r7">Roy K, Kar S, Ambure P. On a simple approach for determining applicability domain of QSAR models. <em>Chemom Intell Lab Syst</em>. 2015;145:22-29. <a href="https://doi.org/10.1016/j.chemolab.2015.04.013">doi:10.1016/j.chemolab.2015.04.013</a></li>
<li id="r8">Vovk V, Gammerman A, Shafer G. <em>Algorithmic Learning in a Random World</em>. Springer; 2005. <a href="https://doi.org/10.1007/b106715">doi:10.1007/b106715</a></li>
<li id="r9">Papadopoulos H, Proedrou K, Vovk V, Gammerman A. Inductive confidence machines for regression. In: <em>ECML 2002</em>. LNCS 2430. Springer; 2002:345-356. <a href="https://doi.org/10.1007/3-540-36755-1_29">doi:10.1007/3-540-36755-1_29</a></li>
<li id="r10">Lei J, G'Sell M, Rinaldo A, Tibshirani RJ, Wasserman L. Distribution-free predictive inference for regression. <em>J Am Stat Assoc</em>. 2018;113(523):1094-1111. <a href="https://doi.org/10.1080/01621459.2017.1307116">doi:10.1080/01621459.2017.1307116</a></li>
<li id="r11">Angelopoulos AN, Bates S. Conformal prediction: a gentle introduction. <em>Found Trends Mach Learn</em>. 2023;16(4):494-591. <a href="https://doi.org/10.1561/2200000101">doi:10.1561/2200000101</a></li>
<li id="r12">Tibshirani RJ, Barber RF, Candes EJ, Ramdas A. Conformal prediction under covariate shift. In: <em>NeurIPS 32</em>; 2019. <a href="https://doi.org/10.48550/arXiv.1904.06019">doi:10.48550/arXiv.1904.06019</a></li>
<li id="r13">Barber RF, Candes EJ, Ramdas A, Tibshirani RJ. Conformal prediction beyond exchangeability. <em>Ann Stat</em>. 2023;51(2):816-845. <a href="https://doi.org/10.1214/23-AOS2276">doi:10.1214/23-AOS2276</a></li>
<li id="r14">Norinder U, Carlsson L, Boyer S, Eklund M. Introducing conformal prediction in predictive modeling: a transparent and flexible alternative to applicability domain determination. <em>J Chem Inf Model</em>. 2014;54(6):1596-1603. <a href="https://doi.org/10.1021/ci5001168">doi:10.1021/ci5001168</a></li>
<li id="r15">Svensson F, Aniceto N, Norinder U, et al. Conformal regression for quantitative structure-activity relationship modeling - quantifying prediction uncertainty. <em>J Chem Inf Model</em>. 2018;58(5):1132-1140. <a href="https://doi.org/10.1021/acs.jcim.8b00054">doi:10.1021/acs.jcim.8b00054</a></li>
<li id="r16">Alvarsson J, McShane SA, Norinder U, Spjuth O. Predicting with confidence: using conformal prediction in drug discovery. <em>J Pharm Sci</em>. 2021;110(1):42-49. <a href="https://doi.org/10.1016/j.xphs.2020.09.055">doi:10.1016/j.xphs.2020.09.055</a></li>
<li id="r17">Mervin LH, Johansson S, Semenova E, Giblin KA, Engkvist O. Uncertainty quantification in drug design. <em>Drug Discov Today</em>. 2021;26(2):474-489. <a href="https://doi.org/10.1016/j.drudis.2020.11.027">doi:10.1016/j.drudis.2020.11.027</a></li>
<li id="r18">Tropsha A. Best practices for QSAR model development, validation, and exploitation. <em>Mol Inform</em>. 2010;29(6-7):476-488. <a href="https://doi.org/10.1002/minf.201000061">doi:10.1002/minf.201000061</a></li>
<li id="r19">Tetko IV, Sushko I, Pandey AK, et al. Critical assessment of QSAR models of environmental toxicity against Tetrahymena pyriformis: focusing on applicability domain and overfitting by variable selection. <em>J Chem Inf Model</em>. 2008;48(9):1733-1746. <a href="https://doi.org/10.1021/ci800151m">doi:10.1021/ci800151m</a></li>
<li id="r20">Koenker R, Bassett G. Regression quantiles. <em>Econometrica</em>. 1978;46(1):33-50. <a href="https://doi.org/10.2307/1913643">doi:10.2307/1913643</a></li>
<li id="r21">Lakshminarayanan B, Pritzel A, Blundell C. Simple and scalable predictive uncertainty estimation using deep ensembles. In: <em>NeurIPS 30</em>; 2017. <a href="https://doi.org/10.48550/arXiv.1612.01474">doi:10.48550/arXiv.1612.01474</a></li>
<li id="r22">Zadrozny B, Elkan C. Transforming classifier scores into accurate multiclass probability estimates. In: <em>Proc. 8th ACM SIGKDD</em>; 2002:694-699. <a href="https://doi.org/10.1145/775047.775151">doi:10.1145/775047.775151</a></li>
<li id="r23">Niculescu-Mizil A, Caruana R. Predicting good probabilities with supervised learning. In: <em>Proc. 22nd Int. Conf. on Machine Learning</em>; 2005:625-632. <a href="https://doi.org/10.1145/1102351.1102430">doi:10.1145/1102351.1102430</a></li>
<li id="r24">Brier GW. Verification of forecasts expressed in terms of probability. <em>Mon Weather Rev</em>. 1950;78(1):1-3. <a href="https://doi.org/10.1175/1520-0493(1950)078%3C0001:VOFEIT%3E2.0.CO;2">doi:10.1175/1520-0493(1950)078&lt;0001:VOFEIT&gt;2.0.CO;2</a></li>
<li id="r25">Van Calster B, McLernon DJ, van Smeden M, Wynants L, Steyerberg EW. Calibration: the Achilles heel of predictive analytics. <em>BMC Med</em>. 2019;17:230. <a href="https://doi.org/10.1186/s12916-019-1466-7">doi:10.1186/s12916-019-1466-7</a></li>
<li id="r26">Sheridan RP. Time-split cross-validation as a method for estimating the goodness of prospective prediction. <em>J Chem Inf Model</em>. 2013;53(4):783-790. <a href="https://doi.org/10.1021/ci400084k">doi:10.1021/ci400084k</a></li>
<li id="r27">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <em>J Med Chem</em>. 1996;39(15):2887-2893. <a href="https://doi.org/10.1021/jm9602928">doi:10.1021/jm9602928</a></li>
<li id="r28">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r29">Breiman L. Random forests. <em>Mach Learn</em>. 2001;45:5-32. <a href="https://doi.org/10.1023/A:1010933404324">doi:10.1023/A:1010933404324</a></li>
<li id="r30">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r31">Wright MN, Ziegler A. ranger: a fast implementation of random forests for high dimensional data in C++ and R. <em>J Stat Softw</em>. 2017;77(1):1-17. <a href="https://doi.org/10.18637/jss.v077.i01">doi:10.18637/jss.v077.i01</a></li>
<li id="r32">Friedman J, Hastie T, Tibshirani R. Regularization paths for generalized linear models via coordinate descent. <em>J Stat Softw</em>. 2010;33(1):1-22. <a href="https://doi.org/10.18637/jss.v033.i01">doi:10.18637/jss.v033.i01</a></li>
<li id="r33">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
<li id="r34">Wallach I, Heifets A. Most ligand-based classification benchmarks reward memorization rather than generalization. <em>J Chem Inf Model</em>. 2018;58(5):916-932. <a href="https://doi.org/10.1021/acs.jcim.7b00403">doi:10.1021/acs.jcim.7b00403</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Vovk V, Gammerman A, Shafer G. <em>Algorithmic Learning in a Random World</em>. 2nd ed. Springer; 2022. - The source text for conformal prediction, including the validity proof sketched here.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. - Chapter 7 for the bias-variance view of prediction error, Chapter 3 for leverage and the hat matrix.</li>
<li>Steyerberg EW. <em>Clinical Prediction Models</em>. 2nd ed. Springer; 2019. - Calibration and decision curves in the clinical setting.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>OECD. <em>Guidance Document on the Validation of (Q)SAR Models</em> and the JRC QMRF template: <a href="https://www.oecd.org/chemicalsafety/risk-assessment/validationofqsarmodels.htm">oecd.org - validation of QSAR models</a>.</li>
<li>Awesome Conformal Prediction, a curated list of tutorials and libraries: <a href="https://github.com/valeman/awesome-conformal-prediction">github.com/valeman/awesome-conformal-prediction</a>.</li>
<li>R packages for conformal prediction (not installed here): <code>conformalInference</code>, <code>cfcausal</code>, <code>crepes</code> (Python): <a href="https://cran.r-project.org/view=MachineLearning">cran.r-project.org/view=MachineLearning</a>.</li>
<li><code>ranger</code> reference manual, including <code>quantreg</code> and <code>predict.all</code>: <a href="https://cran.r-project.org/package=ranger">cran.r-project.org/package=ranger</a>.</li>
</ul>
